# Principio de Sustitución de Liskov (LSP) — Sistema de citas médicas

## Introducción
El Principio de Sustitución de Liskov (LSP) establece que un objeto de una subclase debe poder **sustituir** a un objeto de su clase base sin alterar el correcto funcionamiento del programa: mismo tipo de retorno, mismas precondiciones (o más débiles) y sin excepciones inesperadas.

## Objetivos
- Mostrar una subclase que **sí** es sustituible por su clase base.
- Mostrar un contraejemplo que **viola** el LSP (cambia el tipo de retorno y fortalece una precondición) y explicar el error resultante.

## Clase base y contrato

`Medico.atender_paciente` siempre recibe un paciente y sus síntomas, y siempre devuelve un objeto `ResultadoConsulta`. Ese es el contrato que cualquier subclase debe respetar.

In [1]:
class ResultadoConsulta:
    def __init__(self, diagnostico: str, requiere_seguimiento: bool) -> None:
        self.diagnostico = diagnostico
        self.requiere_seguimiento = requiere_seguimiento

    def __repr__(self) -> str:
        return f"ResultadoConsulta(diagnostico={self.diagnostico!r}, requiere_seguimiento={self.requiere_seguimiento})"


class Medico:
    def __init__(self, nombre: str, especialidad: str) -> None:
        self.nombre = nombre
        self.especialidad = especialidad

    def atender_paciente(self, paciente: str, sintomas: str) -> ResultadoConsulta:
        return ResultadoConsulta(diagnostico=f"Revisión general por: {sintomas}", requiere_seguimiento=False)

    def presentarse(self) -> str:
        return f"Dr(a). {self.nombre} - {self.especialidad}"


class ClinicaVirtual:
    """Clase de alto nivel: agenda y atiende usando cualquier Medico, sin conocer subclases concretas."""

    def __init__(self, medico: Medico) -> None:
        self.medico = medico
        self.consultas_atendidas = 0

    def agendar_y_atender(self, paciente: str, sintomas: str) -> ResultadoConsulta:
        resultado = self.medico.atender_paciente(paciente, sintomas)
        self.consultas_atendidas += 1
        return resultado

    def total_atendidas(self) -> int:
        return self.consultas_atendidas

## Caso correcto: subclase sustituible

`MedicoEspecialista` sobrescribe `atender_paciente` pero mantiene el mismo tipo de retorno (`ResultadoConsulta`), acepta los mismos parámetros y no lanza excepciones nuevas ni inesperadas.

In [2]:
class MedicoEspecialista(Medico):
    def atender_paciente(self, paciente: str, sintomas: str) -> ResultadoConsulta:
        return ResultadoConsulta(
            diagnostico=f"Diagnóstico especializado en {self.especialidad} para: {sintomas}",
            requiere_seguimiento=True,
        )

In [3]:
medico_general = Medico(nombre="Ana Ríos", especialidad="Medicina General")
especialista = MedicoEspecialista(nombre="Carlos Vera", especialidad="Cardiología")

clinica_general = ClinicaVirtual(medico=medico_general)
clinica_especialista = ClinicaVirtual(medico=especialista)

resultado_1 = clinica_general.agendar_y_atender(paciente="Pedro", sintomas="dolor de cabeza")
resultado_2 = clinica_especialista.agendar_y_atender(paciente="Marta", sintomas="dolor en el pecho")

print(resultado_1)
print(resultado_2)

# Ambos resultados son ResultadoConsulta: MedicoEspecialista es sustituible por Medico.
assert isinstance(resultado_1, ResultadoConsulta)
assert isinstance(resultado_2, ResultadoConsulta)
assert resultado_2.requiere_seguimiento is True
print("¡Todo correcto! MedicoEspecialista respeta el contrato de Medico.")

ResultadoConsulta(diagnostico='Revisión general por: dolor de cabeza', requiere_seguimiento=False)
ResultadoConsulta(diagnostico='Diagnóstico especializado en Cardiología para: dolor en el pecho', requiere_seguimiento=True)
¡Todo correcto! MedicoEspecialista respeta el contrato de Medico.


## Contraejemplo: subclase que viola el LSP

`MedicoResidente` rompe el contrato de dos formas:

1. **Fortalece la precondición**: lanza una excepción si `sintomas` está vacío, algo que `Medico` nunca exige.
2. **Cambia el tipo de retorno**: devuelve un `str` en vez de un `ResultadoConsulta`.

In [4]:
class MedicoResidente(Medico):
    def atender_paciente(self, paciente: str, sintomas: str) -> str:
        if sintomas == "":
            # Precondición más fuerte que la de Medico: el padre nunca exige síntomas no vacíos.
            raise ValueError("El residente requiere una descripción de síntomas obligatoria")
        return f"Diagnóstico preliminar: {sintomas}"  # Debería devolver ResultadoConsulta, no str

### Uso que expone la violación

`ClinicaVirtual` fue escrita asumiendo el contrato de `Medico` (siempre devuelve `ResultadoConsulta`). Al sustituir por `MedicoResidente`, el código de alto nivel falla en tiempo de ejecución.

In [5]:
residente = MedicoResidente(nombre="Julián Paz", especialidad="Medicina Interna")
clinica_residente = ClinicaVirtual(medico=residente)

resultado_residente = clinica_residente.agendar_y_atender(paciente="Sofía", sintomas="fiebre")
print(resultado_residente)  # Es un str, no un ResultadoConsulta

try:
    # ClinicaVirtual (y cualquier otro código de alto nivel) espera poder leer .requiere_seguimiento
    # en cualquier resultado, porque ese es el contrato declarado por Medico.
    print(resultado_residente.requiere_seguimiento)
except AttributeError as error:
    print(f"Error al tratar el resultado como ResultadoConsulta: {error}")

Diagnóstico preliminar: fiebre
Error al tratar el resultado como ResultadoConsulta: 'str' object has no attribute 'requiere_seguimiento'


### Análisis del error

`resultado_residente` es un `str`, así que `resultado_residente.requiere_seguimiento` lanza `AttributeError: 'str' object has no attribute 'requiere_seguimiento'`. Cualquier código que dependa del contrato de `Medico` (como `ClinicaVirtual` u otro módulo que procese el resultado) se rompe al recibir un `MedicoResidente` en lugar de un `Medico`.

Además, si `sintomas` llega vacío, `MedicoResidente` lanza `ValueError`, una excepción que `Medico.atender_paciente` nunca declara ni produce — un fortalecimiento de precondición que ningún llamador de `Medico` está preparado para manejar.

Por ambas razones, `MedicoResidente` **no es sustituible** por `Medico`: viola el LSP.

## Autoevaluación
- ¿Qué cambio mínimo haría a `MedicoResidente` para que sí sea sustituible por `Medico`?
- ¿Por qué fortalecer una precondición en una subclase es tan peligroso como cambiar el tipo de retorno?

## Referencias
- Liskov, B. — *A Behavioral Notion of Subtyping* (1994).
- [SOLID Principles en Python – Real Python](https://realpython.com/solid-principles-python/)